# Nested Tuning and Application Model Selection (Analysis D)

## Protocol

**Status:** written on 2026-10-16, before any analysis code in this notebook was run. Under the project's commit policy it is not committed separately before execution; it is committed with the completed work package.

### 1. Framing

This is a **new nested cross-validation analysis** with two purposes:

1. **Research:** estimate how tabular and hybrid models perform when their hyperparameters are tuned without access to the held-out donors, and compare them with the fitted ECD tree.
2. **Application:** select, by a rule fixed here, the model to deploy in the grading service.

Limitations stated in advance:

- The outer folds are the same donor-grouped folds used in notebooks 09–12. They have influenced earlier decisions (cohort, comparators, PCA size), so this analysis is **not an untouched confirmation**. Independent confirmation requires new tissues (notebook 09, Section 9).
- Tuning uses only training-fold data, so the estimate is of the **tuning procedure** on this dataset.

### 2. Cohort

| Set | Use | Expected |
|---|---|---|
| Primary cohort K3: non-2023, image-matched, year from `Tissue_ID` | Training and inner tuning for tabular and hybrid | 551 tissues, 312 donors |
| ECD evaluation set K4: K3 with ECD | Outer-fold evaluation of all models; training of the ECD tree | 470 tissues, 280 donors; A+ 244, A 159, B 67 |

The notebook asserts these counts and stops if they differ.

### 3. Validation design

- **Outer folds:** `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to K3 (25 outer folds).
- **Inner folds:** within each outer training fold, `StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)`, grouped by inferred donor and stratified by grade. Every inner validation fold must contain all three grades, or the notebook stops.
- **Inner selection metric:** macro F1 averaged over the 3 inner validation folds, scored on all inner validation tissues of K3.
- **Leakage controls:** all preprocessing (imputation, scaling, encoding, PCA) is fitted on inner-training data for selection and on the full outer-training fold for the refit. The outer test fold is used only for the final prediction.

### 4. Models and search spaces

All Random Forests use `n_estimators=300, random_state=42, n_jobs=-1`.

| Model | Inputs | Search space | Settings |
|---|---|---|---:|
| Tuned tabular | 24 features (`ms.make_preprocessor()`) | `min_samples_leaf` {1, 2, 5} × `max_features` {"sqrt", 0.3} × `max_depth` {None, 15} × `class_weight` {None, "balanced"} | 24 |
| Tuned hybrid | 24 features + frozen ResNet-50 embedding → StandardScaler → PCA | The tabular grid × PCA components {8, 16, 32} | 72 |
| ECD tree | ECD | None: fixed as in notebook 11 (`max_depth=2, min_samples_leaf=10, class_weight="balanced"`) | 1 |

- `min_samples_split` is fixed at 2 (scikit-learn default).
- **Tie-breaking:** if two settings have the same mean inner macro F1, the first in this order is chosen: fewer PCA components, larger `min_samples_leaf`, `max_depth=15` before None, "sqrt" before 0.3, "balanced" before None.
- **Refit:** the selected setting for each outer fold is refitted on the full outer-training fold (K3 tissues) and predicts the outer test fold.
- **Reproducibility check:** the ECD tree's outer-fold scores must match `fitted_ecd_fold_scores.csv` from notebook 11 (absolute difference < 1e-9).

### 5. Comparisons

Evaluated on the K4 tissues in each outer test fold, with paired fold-level differences, the corrected repeated-CV t-test (24 df) and corrected 95% CIs.

| ID | Comparison | Role |
|---|---|---|
| T1 | Tuned hybrid − ECD tree | Primary |
| T2 | Tuned tabular − ECD tree | Primary |
| T3 | Tuned hybrid − tuned tabular | Primary |

- **Primary metric:** macro F1, with Holm correction across T1–T3.
- **Secondary metrics (unadjusted):** balanced accuracy, accuracy, grade B recall.
- **Descriptive, no tests:**
  - tuning gain: tuned minus untuned scores from notebook 11 for each arm;
  - frequency of each selected setting across the 25 outer folds;
  - per-grade precision and recall, pooled over repeats.

### 6. Application model-selection rule

Fixed before running and applied in code.

1. A candidate (tuned tabular or tuned hybrid) **convincingly outperforms the ECD tree** if its comparison with the tree (T1 for the hybrid, T2 for the tabular model) has macro F1 difference > 0 with Holm-adjusted p < 0.05, **and** its mean fold-level grade B recall difference from the ECD tree is ≥ −0.02.
2. If neither candidate convincingly outperforms the ECD tree, **select the ECD tree**.
3. If exactly one does, select it.
4. If both do, select the hybrid only if T3 has macro F1 difference > 0 with Holm-adjusted p < 0.05 and a grade B recall difference ≥ −0.02 against the tabular model; otherwise select the tabular model (simpler).

The −0.02 threshold is a project-defined decision rule, not a clinically validated threshold.

**Final application configuration:**

- **ECD tree:** fitted on all K3 tissues with ECD; its cut points are reported.
- **Tabular or hybrid:** the same inner tuning procedure (3 grouped folds) is run once on all of K3 to choose the setting, which is then refitted on all of K3.
- Export, the inference pipeline and service integration are implemented separately, in `scripts/` and `ml-service`.

### 7. Interpretation rules

- Results are reported as a **nested-CV analysis on folds that informed earlier decisions**, not as independent confirmation.
- A selected hybrid or tabular model may be described as outperforming a fitted ECD threshold model in this nested analysis only. If the ECD tree is selected, report that tuning did not establish improvement beyond ECD alone.
- Model probabilities are not described as calibrated confidence; no calibration is assessed here.
- **No changes after results:** no other search spaces, learners, inner designs, metrics or selection rules. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
ECD = "ECD_CD_cells_per_mm2"
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB11_SCORES_PATH = ms.PROCESSED_DIR / "fitted_ecd_fold_scores.csv"
SELECTION_PATH = ms.PROCESSED_DIR / "selected_grading_model.json"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
INNER_SPLITS = 3

# Protocol Section 4. Grid order implements the tie-break rule:
# fewer PCA components, larger leaf, depth 15 before None, sqrt before 0.3, balanced before None.
TAB_GRID = [
    {"min_samples_leaf": leaf, "max_depth": depth, "max_features": features, "class_weight": weight}
    for leaf in [5, 2, 1]
    for depth in [15, None]
    for features in ["sqrt", 0.3]
    for weight in ["balanced", None]
]
PCA_GRID = [8, 16, 32]
TREE_PARAMS = dict(max_depth=2, min_samples_leaf=10, class_weight="balanced", criterion="gini", random_state=SEED)
assert len(TAB_GRID) == 24

for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB11_SCORES_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
fitted_ecd_fold_scores.csv: sha256 fe0c10e5d3431501…


## Cohort (Protocol Section 2)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]

primary_desc = {"tissues": len(cohort), "donors": cohort["Donor_Group"].nunique()}
eval_desc = {
    "tissues": len(evaluation), "donors": evaluation["Donor_Group"].nunique(),
    "grades": {k: int(v) for k, v in evaluation[ms.TARGET].value_counts().items()},
}
print("K3:", primary_desc, "| K4:", eval_desc)
assert primary_desc == {"tissues": 551, "donors": 312}, "K3 differs from protocol"
assert eval_desc == {"tissues": 470, "donors": 280, "grades": {"A+": 244, "A": 159, "B": 67}}, "K4 differs from protocol"

cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
donors = cohort["Donor_Group"].to_numpy()
print("Feature matrix:", X.shape)

K3: {'tissues': 551, 'donors': 312} | K4: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}}
Feature matrix: (551, 2072)


## Tuning procedure (Protocol Sections 3–4)

In [3]:
def make_preprocessor(n_components):
    branches = [("tabular", ms.make_preprocessor(), ms.ALL_FEATURES)]
    if n_components:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=SEED)),
        ]), EMBEDDING_COLUMNS))
    return ColumnTransformer(branches, remainder="drop")


def make_forest(setting):
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, **setting)


def macro_f1(true, pred):
    return f1_score(true, pred, labels=ms.CLASSES, average="macro", zero_division=0)


def tune(rows, pca_options):
    """Select a setting by 3-fold donor-grouped inner CV on `rows` (positions in the cohort)."""
    candidates = [(k, s) for k in pca_options for s in TAB_GRID]
    scores = np.zeros((INNER_SPLITS, len(candidates)))
    splitter = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    for i, (inner_train, inner_val) in enumerate(splitter.split(rows, y[rows], groups=donors[rows])):
        tr, va = rows[inner_train], rows[inner_val]
        assert set(y[va]) == set(ms.CLASSES), "An inner validation fold lacks a grade"
        assert set(donors[tr]).isdisjoint(donors[va]), "Inner donor leak"
        for k in pca_options:
            pre = make_preprocessor(k).fit(X.iloc[tr], y[tr])
            Z_tr, Z_va = pre.transform(X.iloc[tr]), pre.transform(X.iloc[va])
            for j, (kk, setting) in enumerate(candidates):
                if kk != k:
                    continue
                scores[i, j] = macro_f1(y[va], make_forest(setting).fit(Z_tr, y[tr]).predict(Z_va))
    mean_scores = scores.mean(axis=0)
    best = int(np.argmax(mean_scores))  # first maximum = tie-break order
    return candidates[best], mean_scores[best]


def fit_selected(rows, choice):
    k, setting = choice
    return Pipeline([("features", make_preprocessor(k)), ("classifier", make_forest(setting))]).fit(X.iloc[rows], y[rows])


ARMS = {"Tuned tabular": [None], "Tuned hybrid": PCA_GRID}

## Outer cross-validation (25 folds)

In [4]:
METRICS = {
    "Macro F1": macro_f1,
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
}
MODELS = ["ECD tree", "Tuned tabular", "Tuned hybrid"]

fold_rows, selections, fold_sizes = [], [], []
oof = {(m, r): np.full(len(cohort), None, dtype=object) for m in MODELS for r in range(ms.N_REPEATS)}

start = time.time()
for repeat in range(ms.N_REPEATS):
    for fold, (train, test) in enumerate(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat)):
        assert set(donors[train]).isdisjoint(donors[test]), "Outer donor leak"
        eval_test = test[has_ecd[test]]
        assert set(y[eval_test]) == set(ms.CLASSES), "An outer test fold lacks a grade"
        fold_sizes.append((len(train), len(test)))

        predictions = {}
        tree = Pipeline([
            ("features", ColumnTransformer([("ecd", "passthrough", [ECD])], remainder="drop")),
            ("classifier", DecisionTreeClassifier(**TREE_PARAMS)),
        ]).fit(X.iloc[train[has_ecd[train]]], y[train[has_ecd[train]]])
        predictions["ECD tree"] = tree.predict(X.iloc[eval_test])

        for arm, pca_options in ARMS.items():
            choice, inner_score = tune(train, pca_options)
            predictions[arm] = fit_selected(train, choice).predict(X.iloc[eval_test])
            selections.append({
                "Arm": arm, "Repeat": repeat, "Fold": fold, "PCA": choice[0],
                **choice[1], "Inner macro F1": inner_score,
            })

        for model, pred in predictions.items():
            oof[(model, repeat)][eval_test] = pred
            fold_rows.append({"Model": model, "Repeat": repeat, "Fold": fold, "N": len(eval_test),
                              **{m: fn(y[eval_test], pred) for m, fn in METRICS.items()}})

        print(f"Repeat {repeat} fold {fold} done — {(time.time() - start) / 60:.1f} min elapsed", flush=True)

fold_scores = pd.DataFrame(fold_rows)
selections = pd.DataFrame(selections)
N_TRAIN, N_TEST = np.mean(fold_sizes, axis=0)
print(f"\nTotal time {(time.time() - start) / 60:.1f} min")

Repeat 0 fold 0 done — 3.4 min elapsed


Repeat 0 fold 1 done — 7.3 min elapsed


Repeat 0 fold 2 done — 11.2 min elapsed


Repeat 0 fold 3 done — 15.2 min elapsed


Repeat 0 fold 4 done — 18.1 min elapsed


...

Repeat 4 fold 1 done — 64.9 min elapsed


Repeat 4 fold 2 done — 67.3 min elapsed


Repeat 4 fold 3 done — 69.8 min elapsed


Repeat 4 fold 4 done — 72.3 min elapsed



Total time 72.3 min


### Reproducibility check (Protocol Section 4)

In [5]:
saved = pd.read_csv(NB11_SCORES_PATH)
a = fold_scores[fold_scores["Model"] == "ECD tree"].set_index(["Repeat", "Fold"])[list(METRICS)].sort_index()
b = saved[saved["Model"] == "ECD tree"].set_index(["Repeat", "Fold"])[list(METRICS)].sort_index()
max_diff = (a - b).abs().to_numpy().max()
print(f"ECD tree: max absolute difference from notebook 11 = {max_diff:.2e}")
assert max_diff < 1e-9, "ECD tree differs from notebook 11: stop and record in the deviations log"

ECD tree: max absolute difference from notebook 11 = 0.00e+00


## Results (Protocol Section 5)

In [6]:
print("Outer-fold metrics on K4 tissues, mean and SD over 25 folds:")
print(fold_scores.groupby("Model")[list(METRICS)].agg(["mean", "std"]).reindex(MODELS).round(4).to_string())

Outer-fold metrics on K4 tissues, mean and SD over 25 folds:
              Macro F1         Balanced accuracy         Accuracy         Recall B        
                  mean     std              mean     std     mean     std     mean     std
Model                                                                                     
ECD tree        0.7958  0.0383            0.7680  0.0431   0.8044  0.0308   0.7901  0.1171
Tuned tabular   0.8049  0.0338            0.7811  0.0366   0.8086  0.0298   0.7931  0.1154
Tuned hybrid    0.7934  0.0466            0.7750  0.0496   0.7971  0.0396   0.7891  0.1300


In [7]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Model", values=list(METRICS))
COMPARISONS = {
    "T1": ("Tuned hybrid", "ECD tree"),
    "T2": ("Tuned tabular", "ECD tree"),
    "T3": ("Tuned hybrid", "Tuned tabular"),
}
rows = []
for label, (model, baseline) in COMPARISONS.items():
    for metric in METRICS:
        d = wide[(metric, model)] - wide[(metric, baseline)]
        mean, low, high, t, p = corrected_ttest(d)
        rows.append({
            "ID": label, "Comparison": f"{model} − {baseline}",
            "Role": "Primary" if metric == "Macro F1" else "Secondary", "Metric": metric,
            "Model": wide[(metric, model)].mean(), "Baseline": wide[(metric, baseline)].mean(),
            "Difference": mean, "CI low": low, "CI high": high, "t": t, "p": p,
            "Folds improved": f"{(d > 0).sum()}/{len(d)}",
        })
results = pd.DataFrame(rows)
primary_mask = results["Role"] == "Primary"
results["p (Holm)"] = np.nan
results.loc[primary_mask, "p (Holm)"] = holm(results.loc[primary_mask, "p"])

print("Primary (macro F1; Holm across T1–T3; CIs not multiplicity-adjusted):")
print(results[primary_mask].drop(columns="Role").round(4).to_string(index=False))
print("\nSecondary (unadjusted):")
print(results[~primary_mask].drop(columns=["Role", "p (Holm)"]).round(4).to_string(index=False))

Primary (macro F1; Holm across T1–T3; CIs not multiplicity-adjusted):
ID                   Comparison   Metric  Model  Baseline  Difference  CI low  CI high       t      p Folds improved  p (Holm)
T1      Tuned hybrid − ECD tree Macro F1 0.7934    0.7958     -0.0025 -0.0341   0.0291 -0.1616 0.8730          11/25       1.0
T2     Tuned tabular − ECD tree Macro F1 0.8049    0.7958      0.0091 -0.0102   0.0284  0.9692 0.3421          18/25       1.0
T3 Tuned hybrid − Tuned tabular Macro F1 0.7934    0.8049     -0.0115 -0.0448   0.0217 -0.7157 0.4811           8/25       1.0

Secondary (unadjusted):
ID                   Comparison            Metric  Model  Baseline  Difference  CI low  CI high       t      p Folds improved
T1      Tuned hybrid − ECD tree Balanced accuracy 0.7750    0.7680      0.0070 -0.0223   0.0363  0.4943 0.6256          12/25
T1      Tuned hybrid − ECD tree          Accuracy 0.7971    0.8044     -0.0073 -0.0364   0.0218 -0.5165 0.6103           8/25
T1      Tuned hybri

### Descriptive results

In [8]:
untuned = saved.set_index(["Repeat", "Fold", "Model"])["Macro F1"].unstack()
for tuned_name, untuned_name in [("Tuned tabular", "Tabular"), ("Tuned hybrid", "Hybrid")]:
    d = wide[("Macro F1", tuned_name)] - untuned[untuned_name]
    print(f"Tuning gain, {untuned_name}: mean fold-level macro F1 difference {d.mean():+.4f} "
          f"(tuned higher in {(d > 0).sum()}/{len(d)} folds; descriptive)")

print("\nSelected settings across 25 outer folds:")
setting_columns = ["PCA", "min_samples_leaf", "max_depth", "max_features", "class_weight"]
for arm in ARMS:
    counts = (selections[selections["Arm"] == arm][setting_columns].astype(str)
              .value_counts().rename("Folds").reset_index())
    print(f"\n{arm}:")
    print(counts.to_string(index=False))

Tuning gain, Tabular: mean fold-level macro F1 difference +0.0492 (tuned higher in 24/25 folds; descriptive)
Tuning gain, Hybrid: mean fold-level macro F1 difference +0.0048 (tuned higher in 15/25 folds; descriptive)

Selected settings across 25 outer folds:

Tuned tabular:
Empty DataFrame
Columns: [PCA, min_samples_leaf, max_depth, max_features, class_weight, Folds]
Index: []

Tuned hybrid:
 PCA min_samples_leaf max_depth max_features class_weight  Folds
32.0                5      15.0          0.3     balanced      6
32.0                2      15.0          0.3     balanced      3
 8.0                1      15.0          0.3     balanced      1
16.0                1      15.0          0.3     balanced      1
32.0                1      15.0          0.3     balanced      1
16.0                2      15.0          0.3     balanced      1


In [9]:
eval_rows = np.flatnonzero(has_ecd)
y_eval = np.tile(y[eval_rows], ms.N_REPEATS)
pooled = {m: np.concatenate([oof[(m, r)][eval_rows] for r in range(ms.N_REPEATS)]).astype(str) for m in MODELS}
print("Per-grade precision and recall on K4, pooled over repeats:")
print(pd.concat({
    m: pd.DataFrame({
        "Precision": precision_score(y_eval, pooled[m], labels=ms.CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_eval, pooled[m], labels=ms.CLASSES, average=None, zero_division=0),
    }, index=ms.CLASSES)
    for m in MODELS
}, axis=1).round(3).to_string())

Per-grade precision and recall on K4, pooled over repeats:
    ECD tree        Tuned tabular        Tuned hybrid       
   Precision Recall     Precision Recall    Precision Recall
A+     0.748  0.988         0.779  0.943        0.788  0.906
A      0.884  0.528         0.805  0.609        0.761  0.635
B      1.000  0.791         0.978  0.794        0.936  0.791


## Application model selection (Protocol Section 6)

The rule is applied mechanically.

In [10]:
primary = results[primary_mask].set_index("ID")
b_recall = results[results["Metric"] == "Recall B"].set_index("ID")["Difference"]


def convincing(comparison_id):
    row = primary.loc[comparison_id]
    return bool(row["Difference"] > 0 and row["p (Holm)"] < 0.05 and b_recall[comparison_id] >= -0.02)


beats_tree = {"Tuned hybrid": convincing("T1"), "Tuned tabular": convincing("T2")}
for arm, ok in beats_tree.items():
    cid = "T1" if arm == "Tuned hybrid" else "T2"
    row = primary.loc[cid]
    print(f"{arm} vs ECD tree: macro F1 {row['Difference']:+.4f} [{row['CI low']:+.4f}, {row['CI high']:+.4f}], "
          f"Holm p = {row['p (Holm)']:.4f}, B recall difference {b_recall[cid]:+.4f} -> "
          f"{'convincingly outperforms' if ok else 'does not convincingly outperform'}")

if not any(beats_tree.values()):
    selected = "ECD tree"
elif sum(beats_tree.values()) == 1:
    selected = next(arm for arm, ok in beats_tree.items() if ok)
else:
    selected = "Tuned hybrid" if convincing("T3") else "Tuned tabular"

print(f"\nSelected application model: {selected}")
if selected == "ECD tree":
    print("Tuning did not establish improvement beyond ECD alone (nested-CV analysis).")

Tuned hybrid vs ECD tree: macro F1 -0.0025 [-0.0341, +0.0291], Holm p = 1.0000, B recall difference -0.0010 -> does not convincingly outperform
Tuned tabular vs ECD tree: macro F1 +0.0091 [-0.0102, +0.0284], Holm p = 1.0000, B recall difference +0.0031 -> does not convincingly outperform

Selected application model: ECD tree
Tuning did not establish improvement beyond ECD alone (nested-CV analysis).


### Final application configuration (Protocol Section 6)

In [11]:
all_rows = np.arange(len(cohort))
if selected == "ECD tree":
    ecd_rows = all_rows[has_ecd]
    final_tree = DecisionTreeClassifier(**TREE_PARAMS).fit(X.iloc[ecd_rows][[ECD]], y[ecd_rows])
    tree = final_tree.tree_
    cut_points = sorted(float(t) for t in tree.threshold[tree.feature >= 0])
    config = {"model": "ecd_tree", "params": TREE_PARAMS, "training_tissues": int(len(ecd_rows)),
              "cut_points": cut_points}
    print(f"ECD tree fitted on {len(ecd_rows)} K3 tissues with ECD; cut points {cut_points}")
    leaves = pd.DataFrame({"Leaf": final_tree.apply(X.iloc[ecd_rows][[ECD]]), "Grade": y[ecd_rows]})
    print("Training grade counts per leaf:")
    print(pd.crosstab(leaves["Leaf"], leaves["Grade"])[ms.CLASSES].to_string())
else:
    choice, inner_score = tune(all_rows, ARMS[selected])
    config = {"model": "hybrid" if selected == "Tuned hybrid" else "tabular", "pca_components": choice[0],
              "forest_params": {**choice[1], "n_estimators": 300, "random_state": SEED},
              "inner_macro_f1": float(inner_score), "training_tissues": int(len(all_rows))}
    print(f"Final setting chosen by inner CV on all K3: {choice} (inner macro F1 {inner_score:.4f})")

config.update({
    "selected_by": "13_nested_tuning_model_selection.ipynb",
    "cohort": "K3: non-2023, image-matched, year from Tissue_ID",
    "data_sha256": hashlib.sha256(ms.DATA_PATH.read_bytes()).hexdigest(),
})
SELECTION_PATH.write_text(json.dumps(config, indent=2, default=str))
print("Saved selection to", SELECTION_PATH)

ECD tree fitted on 470 K3 tissues with ECD; cut points [2232.0, 2497.0]
Training grade counts per leaf:
Grade   A+   A   B
Leaf              
1        0   0  53
3        2  84   8
4      242  75   6
Saved selection to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed\selected_grading_model.json


In [12]:
fold_scores.to_csv(ms.PROCESSED_DIR / "nested_tuning_fold_scores.csv", index=False)
results.to_csv(ms.PROCESSED_DIR / "nested_tuning_results.csv", index=False)
selections.to_csv(ms.PROCESSED_DIR / "nested_tuning_selections.csv", index=False)
print("Saved fold scores, comparisons and selected settings to", ms.PROCESSED_DIR)

Saved fold scores, comparisons and selected settings to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Post-run correction: selected settings

The "Selected settings across 25 outer folds" output above is **incomplete**. In pandas 3, `astype(str)` keeps missing values (`max_depth=None`, `class_weight=None`, and `PCA=None` for the tabular arm) as missing, so `value_counts()` dropped every row containing one. The tabular table was empty and the hybrid table showed 13 of 25 folds.

The tables below are rebuilt from the saved `nested_tuning_selections.csv`, which the notebook wrote from the same in-memory selections. No model was refitted, and no other result depends on that output.

**Tuned tabular** (25 outer folds)

| PCA | min_samples_leaf | max_depth | max_features | class_weight | Folds |
|---|---|---|---|---|---|
| None | 5 | 15 | 0.3 | None | 8 |
| None | 2 | 15 | 0.3 | None | 6 |
| None | 1 | 15 | 0.3 | None | 4 |
| None | 1 | None | 0.3 | None | 4 |
| None | 5 | 15 | 0.3 | balanced | 1 |
| None | 2 | None | 0.3 | None | 1 |
| None | 2 | 15 | 0.3 | balanced | 1 |

**Tuned hybrid** (25 outer folds)

| PCA | min_samples_leaf | max_depth | max_features | class_weight | Folds |
|---|---|---|---|---|---|
| 32 | 5 | 15 | 0.3 | balanced | 6 |
| 32 | 2 | 15 | 0.3 | balanced | 3 |
| 32 | 1 | 15 | 0.3 | None | 3 |
| 16 | 1 | None | 0.3 | balanced | 2 |
| 8 | 1 | 15 | 0.3 | balanced | 1 |
| 16 | 1 | 15 | 0.3 | balanced | 1 |
| 16 | 1 | 15 | 0.3 | None | 1 |
| 16 | 2 | 15 | 0.3 | None | 1 |
| 8 | 1 | 15 | 0.3 | None | 1 |
| 32 | 1 | 15 | 0.3 | balanced | 1 |
| 32 | 1 | None | 0.3 | None | 1 |
| 32 | 1 | None | 0.3 | balanced | 1 |
| 16 | 2 | 15 | 0.3 | balanced | 1 |
| 16 | 1 | None | 0.3 | None | 1 |
| 32 | 2 | 15 | 0.3 | None | 1 |

Every fold in both arms selected `max_features=0.3`. The tabular arm mostly selected no class weighting (22 of 25 folds).

## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| 2026-10-16 | Removed saved stderr warning streams from cell outputs (the notebook file was 87 MB). No code was re-executed. | scikit-learn's joblib worker processes emitted a `UserWarning` on every fit; the main-process warning filter did not reach them. Cosmetic only. |
| 2026-10-16 | Added the "Post-run correction: selected settings" cell with tables rebuilt from `nested_tuning_selections.csv`. The original output was left in place. | Reporting bug: pandas 3 `astype(str)` kept missing values, so `value_counts()` dropped rows. No result, comparison or selection depends on this table. |